In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = os.getcwd() if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Mounted at /content/drive
GitHub Token: ··········
Cloning into 'RecSys-Challenge-2025'...
remote: Enumerating objects: 278, done.
remote: Counting objects: 100% (278/278), done.
remote: Compressing objects: 100% (211/211), done.
remote: Total 278 (delta 105), reused 227 (delta 59), pack-reused 0 (from 0)
Receiving objects: 100% (278/278), 7.39 MiB | 19.87 MiB/s, done.
Resolving deltas: 100% (105/105), done.


In [12]:
import importlib
import scipy.sparse as sps
import pandas as pd

from Challenge import paths
importlib.reload(paths)

Running on colab — storage at: /content/drive/MyDrive/RecSys


<module 'Challenge.paths' from '/content/RecSys-Challenge-2025/Challenge/paths.py'>

In [3]:
# Load datasets
URM_train = sps.load_npz(paths.URM_TRAIN)
URM_validation = sps.load_npz(paths.URM_VALIDATION)

In [4]:
from Evaluation.Evaluator import EvaluatorHoldout

# Set up evaluator
evaluator = EvaluatorHoldout(URM_validation, cutoff_list=[20])

EvaluatorHoldout: Ignoring 38 ( 0.1%) Users that have less than 1 test interactions


In [5]:
from Recommenders.NonPersonalizedRecommender import TopPop

# Create recommender
recommender = TopPop(URM_train)
recommender.fit()

In [6]:
results_df, results_run_string = evaluator.evaluateRecommender(recommender)
print("RECALL@20: ", results_df.loc[20]["RECALL"])

EvaluatorHoldout: Processed 27057 (100.0%) in 14.63 sec. Users per second: 1849


In [11]:
# Train final model on train + validation with best hyperparameters
recommender = TopPop(URM_train + URM_validation)
recommender.fit()

# Save the trained model
recommender.save_model(paths.MODEL_DIR)

TopPopRecommender: Saving model in file '/content/drive/MyDrive/RecSys/modelsTopPopRecommender'
TopPopRecommender: Saving complete


In [14]:
# Generate recommendations for the test set
user_ids_test = pd.read_csv(paths.CHALLENGE_USER_IDS_TEST)
ids = user_ids_test["user_id"].values

recommendations = recommender.recommend(ids, cutoff=20)

In [17]:
submission_name = "TopPop.csv"

os.makedirs(paths.SUBMISSIONS, exist_ok=True)
with open(os.path.join(paths.SUBMISSIONS, submission_name), "w") as f:
    f.write("user_id,item_list\n")
    for user_id, rec_list in zip(ids, recommendations):
        f.write(f"{user_id},{' '.join([str(item) for item in rec_list])}\n")